# Model Training Notebook

This notebook trains and compares multiple regression models for salary prediction. It keeps `salary_in_usd` as the target, uses the same main project features, compares every model against a median baseline, and saves the best real-performing pipeline for the API.


In [1]:
import json
from pathlib import Path

import joblib
import numpy as np
import pandas as pd

from sklearn.base import clone
from sklearn.compose import ColumnTransformer, TransformedTargetRegressor
from sklearn.ensemble import GradientBoostingRegressor, RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder
from sklearn.tree import DecisionTreeRegressor

pd.set_option("display.max_columns", None)
pd.set_option("display.width", 120)

RANDOM_STATE = 42


## Load Cleaned Data

Use the processed dataset created by `01_data_cleaning.ipynb`.


In [2]:
data_path = Path("../data/processed/cleaned_salaries.csv")
df = pd.read_csv(data_path)

print(f"Training dataset shape: {df.shape}")
print(df.head())
print(df["salary_in_usd"].describe())


Training dataset shape: (558, 10)
   work_year experience_level employment_type                   job_title employee_residence  remote_ratio  \
0       2020        Mid-level       Full-time              Data Scientist                 DE             0   
1       2020     Senior-level       Full-time  Machine Learning Scientist                 JP             0   
2       2020     Senior-level       Full-time           Big Data Engineer                 GB            50   
3       2020        Mid-level       Full-time                       Other                 HN             0   
4       2020     Senior-level       Full-time   Machine Learning Engineer                 US            50   

  work_setting company_location company_size  salary_in_usd  
0      On-site               DE        Large          79833  
1      On-site               JP        Small         260000  
2       Hybrid               GB       Medium         109024  
3      On-site               HN        Small          200

## Select Features and Target

The target stays `salary_in_usd`. `work_setting` is not used as a feature because it duplicates `remote_ratio`.


In [3]:
target_column = "salary_in_usd"
feature_columns = [
    "work_year",
    "experience_level",
    "employment_type",
    "job_title",
    "employee_residence",
    "remote_ratio",
    "company_location",
    "company_size",
]

X = df[feature_columns].copy()
y = df[target_column].copy()

numeric_features = ["work_year", "remote_ratio"]
categorical_features = [
    "experience_level",
    "employment_type",
    "job_title",
    "employee_residence",
    "company_location",
    "company_size",
]

print(f"Features: {feature_columns}")
print(f"Target: {target_column}")


Features: ['work_year', 'experience_level', 'employment_type', 'job_title', 'employee_residence', 'remote_ratio', 'company_location', 'company_size']
Target: salary_in_usd


## Train/Test Split

The test set is held back for final evaluation only. No model is fitted on the test rows.


In [4]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=RANDOM_STATE
)

print(f"Train rows: {X_train.shape[0]}")
print(f"Test rows: {X_test.shape[0]}")


Train rows: 446
Test rows: 112


## Baseline

The baseline predicts the median training salary for every test row. A useful model should beat this on MAE and preferably R2.


In [5]:
def evaluate_predictions(y_true, y_pred):
    return {
        "mae": float(mean_absolute_error(y_true, y_pred)),
        "rmse": float(np.sqrt(mean_squared_error(y_true, y_pred))),
        "r2": float(r2_score(y_true, y_pred)),
    }

baseline_value = float(y_train.median())
baseline_predictions = np.full(shape=len(y_test), fill_value=baseline_value)
baseline_metrics = evaluate_predictions(y_test, baseline_predictions)

print(f"Baseline median salary: ${baseline_value:,.2f}")
print(baseline_metrics)


Baseline median salary: $100,000.00
{'mae': 49323.919642857145, 'rmse': 59810.1395961331, 'r2': -0.014313700157837506}


## Preprocessing and Candidate Models

Each model uses the same `ColumnTransformer`: one-hot encoding for categorical features and passthrough for numeric features.


In [6]:
def make_one_hot_encoder():
    try:
        return OneHotEncoder(handle_unknown="ignore", sparse_output=False)
    except TypeError:
        return OneHotEncoder(handle_unknown="ignore", sparse=False)


def build_preprocessor():
    return ColumnTransformer(
        transformers=[
            ("categorical", make_one_hot_encoder(), categorical_features),
            ("numeric", "passthrough", numeric_features),
        ]
    )


def build_pipeline(estimator):
    return Pipeline(
        steps=[
            ("preprocessor", build_preprocessor()),
            ("model", estimator),
        ]
    )

models = {
    "DecisionTreeRegressor": DecisionTreeRegressor(
        max_depth=7, min_samples_split=5, min_samples_leaf=2, random_state=RANDOM_STATE
    ),
    "RandomForestRegressor": RandomForestRegressor(
        n_estimators=300, min_samples_leaf=2, random_state=RANDOM_STATE, n_jobs=-1
    ),
    "GradientBoostingRegressor": GradientBoostingRegressor(
        n_estimators=250, learning_rate=0.05, max_depth=3, random_state=RANDOM_STATE
    ),
}

print(list(models.keys()))


['DecisionTreeRegressor', 'RandomForestRegressor', 'GradientBoostingRegressor']


## Train and Evaluate Models

Each candidate is trained once on the raw target and once with `np.log1p(salary_in_usd)`. For log models, predictions are converted back to dollars with `np.expm1` before evaluation.


In [7]:
results = []
trained_models = {}

for model_name, estimator in models.items():
    raw_pipeline = build_pipeline(clone(estimator))
    raw_pipeline.fit(X_train, y_train)
    raw_predictions = raw_pipeline.predict(X_test)
    raw_metrics = evaluate_predictions(y_test, raw_predictions)
    raw_key = f"{model_name} | raw target"
    trained_models[raw_key] = raw_pipeline
    results.append({
        "candidate": raw_key,
        "model_name": model_name,
        "target_transform": "none",
        **raw_metrics,
    })

    log_pipeline = build_pipeline(clone(estimator))
    log_model = TransformedTargetRegressor(
        regressor=log_pipeline,
        func=np.log1p,
        inverse_func=np.expm1,
        check_inverse=False,
    )
    log_model.fit(X_train, y_train)
    log_predictions = np.maximum(log_model.predict(X_test), 0)
    log_metrics = evaluate_predictions(y_test, log_predictions)
    log_key = f"{model_name} | log target"
    trained_models[log_key] = log_model
    results.append({
        "candidate": log_key,
        "model_name": model_name,
        "target_transform": "log1p/expm1",
        **log_metrics,
    })

comparison_df = pd.DataFrame(results)
comparison_df = comparison_df.sort_values(["mae", "r2"], ascending=[True, False]).reset_index(drop=True)

print("Model comparison table:")
print(comparison_df.to_string(index=False, formatters={
    "mae": "{:,.2f}".format,
    "rmse": "{:,.2f}".format,
    "r2": "{:.4f}".format,
}))


Model comparison table:
                             candidate                model_name target_transform       mae      rmse     r2
    RandomForestRegressor | raw target     RandomForestRegressor             none 24,817.84 33,966.23 0.6729
GradientBoostingRegressor | raw target GradientBoostingRegressor             none 24,851.36 33,410.29 0.6835
    DecisionTreeRegressor | raw target     DecisionTreeRegressor             none 25,939.47 35,717.39 0.6383
    RandomForestRegressor | log target     RandomForestRegressor      log1p/expm1 26,243.23 35,483.01 0.6430
GradientBoostingRegressor | log target GradientBoostingRegressor      log1p/expm1 26,536.27 37,687.18 0.5973
    DecisionTreeRegressor | log target     DecisionTreeRegressor      log1p/expm1 29,851.84 39,401.83 0.5598


## Select Best Model

The best model is chosen mainly by lower MAE, with R2 used as a secondary check. The selected model must be compared against the median baseline honestly.


In [8]:
best_row = comparison_df.iloc[0].to_dict()
best_candidate = best_row["candidate"]
best_model = trained_models[best_candidate]

print(f"Best model: {best_candidate}")
print("Best model metrics:")
print({
    "mae": best_row["mae"],
    "rmse": best_row["rmse"],
    "r2": best_row["r2"],
})
print("Baseline metrics:")
print(baseline_metrics)
print(f"MAE improvement vs baseline: ${baseline_metrics['mae'] - best_row['mae']:,.2f}")
print(f"R2 improvement vs baseline: {best_row['r2'] - baseline_metrics['r2']:.4f}")


Best model: RandomForestRegressor | raw target
Best model metrics:
{'mae': 24817.84344509897, 'rmse': 33966.23098234126, 'r2': 0.6728722574803161}
Baseline metrics:
{'mae': 49323.919642857145, 'rmse': 59810.1395961331, 'r2': -0.014313700157837506}
MAE improvement vs baseline: $24,506.08
R2 improvement vs baseline: 0.6872


## Sample Prediction

Run one valid input through the selected pipeline to confirm the saved model predicts salaries in USD.


In [9]:
sample_input = pd.DataFrame([
    {
        "work_year": 2022,
        "experience_level": "Senior-level",
        "employment_type": "Full-time",
        "job_title": "Data Scientist",
        "employee_residence": "US",
        "remote_ratio": 100,
        "company_location": "US",
        "company_size": "Medium",
    }
])

sample_prediction = float(best_model.predict(sample_input)[0])
print(f"Sample predicted salary: ${sample_prediction:,.2f}")


Sample predicted salary: $165,977.67


## Save Model, Metrics, and Schema

Persist the best trained pipeline and metadata used by the API and dashboard.


In [10]:
models_dir = Path("../models")
models_dir.mkdir(parents=True, exist_ok=True)

model_path = models_dir / "salary_best_model_pipeline.joblib"
joblib.dump(best_model, model_path)
print(f"Best model saved to: {model_path}")

metrics = {
    "model_name": best_row["model_name"],
    "candidate": best_candidate,
    "target": target_column,
    "features": feature_columns,
    "target_transform": best_row["target_transform"],
    "mae": best_row["mae"],
    "rmse": best_row["rmse"],
    "r2": best_row["r2"],
    "baseline_strategy": "median_training_salary",
    "baseline_salary": baseline_value,
    "baseline_mae": baseline_metrics["mae"],
    "baseline_rmse": baseline_metrics["rmse"],
    "baseline_r2": baseline_metrics["r2"],
    "test_size": 0.2,
    "random_state": RANDOM_STATE,
    "salary_outlier_method": "Removed salary_in_usd values above the 99th percentile in cleaning notebook",
    "rare_job_title_threshold": 5,
    "model_comparison": comparison_df.to_dict(orient="records"),
}

metrics_path = models_dir / "model_metrics.json"
with open(metrics_path, "w", encoding="utf-8") as file:
    json.dump(metrics, file, indent=4)
print(f"Metrics saved to: {metrics_path}")

input_schema = {
    "work_year": sorted(int(value) for value in X["work_year"].dropna().unique()),
    "experience_level": sorted(X["experience_level"].dropna().unique().tolist()),
    "employment_type": sorted(X["employment_type"].dropna().unique().tolist()),
    "job_title": sorted(X["job_title"].dropna().unique().tolist()),
    "employee_residence": sorted(X["employee_residence"].dropna().unique().tolist()),
    "remote_ratio": sorted(int(value) for value in X["remote_ratio"].dropna().unique()),
    "company_location": sorted(X["company_location"].dropna().unique().tolist()),
    "company_size": sorted(X["company_size"].dropna().unique().tolist()),
}

schema_path = models_dir / "input_schema.json"
with open(schema_path, "w", encoding="utf-8") as file:
    json.dump(input_schema, file, indent=4)
print(f"Input schema saved to: {schema_path}")

api_allowed_values_path = Path("../api/allowed_values.json")
with open(api_allowed_values_path, "w", encoding="utf-8") as file:
    json.dump(input_schema, file, indent=4)
print(f"API allowed values saved to: {api_allowed_values_path}")


Best model saved to: ..\models\salary_best_model_pipeline.joblib
Metrics saved to: ..\models\model_metrics.json
Input schema saved to: ..\models\input_schema.json
API allowed values saved to: ..\api\allowed_values.json


## Save Feature Importance

Tree-based models expose `feature_importances_`. Save the encoded feature importance table for the selected model.


In [11]:
def unwrap_pipeline(model):
    if isinstance(model, TransformedTargetRegressor):
        return model.regressor_
    return model

fitted_pipeline = unwrap_pipeline(best_model)
fitted_preprocessor = fitted_pipeline.named_steps["preprocessor"]
fitted_estimator = fitted_pipeline.named_steps["model"]

feature_importance_path = Path("../data/processed/feature_importance.csv")

if hasattr(fitted_estimator, "feature_importances_"):
    feature_names = fitted_preprocessor.get_feature_names_out()
    feature_importance_df = pd.DataFrame({
        "feature": feature_names,
        "importance": fitted_estimator.feature_importances_,
    }).sort_values("importance", ascending=False)
    feature_importance_df.to_csv(feature_importance_path, index=False)
    print(f"Feature importance saved to: {feature_importance_path}")
    print(feature_importance_df.head(20).to_string(index=False))
else:
    feature_importance_df = pd.DataFrame(columns=["feature", "importance"])
    feature_importance_df.to_csv(feature_importance_path, index=False)
    print("Selected model does not expose feature_importances_. Empty file saved.")


Feature importance saved to: ..\data\processed\feature_importance.csv
                                         feature  importance
              categorical__employee_residence_US    0.428698
             categorical__job_title_Data Analyst    0.058773
   categorical__experience_level_Executive-level    0.058178
      categorical__experience_level_Senior-level    0.054574
       categorical__experience_level_Entry-level    0.040867
                           numeric__remote_ratio    0.030207
                              numeric__work_year    0.029702
         categorical__experience_level_Mid-level    0.025219
            categorical__job_title_Data Engineer    0.021912
                    categorical__job_title_Other    0.020038
           categorical__job_title_Data Scientist    0.015449
   categorical__job_title_Applied Data Scientist    0.014531
                categorical__company_size_Medium    0.013310
                 categorical__company_size_Large    0.012762
 categorical__j